# RoPE notebook

A notebook-local RoPE implementation with one unified class entrypoint and two test cells.


In [ ]:
from dataclasses import dataclass
from typing import Optional

import torch
from torch import nn


@dataclass
class RoPEConfig:
    hidden_size: int
    num_attention_heads: int
    max_position_embeddings: int
    rope_theta: float = 10000.0
    head_dim: Optional[int] = None


class LocalLlamaRoPE(nn.Module):
    def __init__(self, config: RoPEConfig, device: Optional[torch.device] = None):
        super().__init__()
        self.config = config
        inv_freq, self.attention_scaling = self.compute_default_rope_parameters(config, device)
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("original_inv_freq", inv_freq.clone(), persistent=False)

    @staticmethod
    def compute_default_rope_parameters(
        config: RoPEConfig,
        device: Optional[torch.device] = None,
        seq_len: Optional[int] = None,
    ) -> tuple[torch.Tensor, float]:
        del seq_len
        dim = config.head_dim or config.hidden_size // config.num_attention_heads
        inv_freq = 1.0 / (
            config.rope_theta
            ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim)
        )
        return inv_freq, 1.0

    @staticmethod
    def rotate_half(x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    @staticmethod
    def apply_rotary_pos_emb(
        q: torch.Tensor,
        k: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        unsqueeze_dim: int = 1,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        cos = cos.unsqueeze(unsqueeze_dim)
        sin = sin.unsqueeze(unsqueeze_dim)
        q_embed = (q * cos) + (LocalLlamaRoPE.rotate_half(q) * sin)
        k_embed = (k * cos) + (LocalLlamaRoPE.rotate_half(k) * sin)
        return q_embed, k_embed

    @torch.no_grad()
    def forward(
        self,
        x: torch.Tensor,
        position_ids: torch.Tensor,
        q: Optional[torch.Tensor] = None,
        k: Optional[torch.Tensor] = None,
        unsqueeze_dim: int = 1,
    ):
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()
        freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        
        cos = (emb.cos() * self.attention_scaling).to(dtype=x.dtype)
        sin = (emb.sin() * self.attention_scaling).to(dtype=x.dtype)

        if q is None and k is None:
            return cos, sin
        if q is None or k is None:
            raise ValueError("q and k must both be provided when applying RoPE.")

        q_embed, k_embed = self.apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=unsqueeze_dim)
        return q_embed, k_embed, cos, sin


def make_config() -> RoPEConfig:
    return RoPEConfig(hidden_size=8, num_attention_heads=2, max_position_embeddings=64)


In [10]:
# Example usage
config = make_config()
rope = LocalLlamaRoPE(config)
x = torch.zeros(1, config.num_attention_heads, 4, config.hidden_size // config.num_attention_heads)
position_ids = torch.arange(4).unsqueeze(0)
cos, sin = rope(x, position_ids)
print("cos shape:", cos.shape)
print("sin shape:", sin.shape)


cos shape: torch.Size([1, 4, 4])
sin shape: torch.Size([1, 4, 4])


In [13]:
# Test case 1: cos/sin generation matches the explicit formula
config = make_config()
rope = LocalLlamaRoPE(config)
x = torch.zeros(1, config.num_attention_heads, 3, config.hidden_size // config.num_attention_heads, dtype=torch.float32)
position_ids = torch.tensor([[0, 1, 2]])
cos, sin = rope(x, position_ids)

dim = config.hidden_size // config.num_attention_heads
inv_freq = 1.0 / (config.rope_theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
freqs = torch.outer(position_ids[0].float(), inv_freq)
expected_emb = torch.cat((freqs, freqs), dim=-1).unsqueeze(0)
expected_cos = expected_emb.cos()
expected_sin = expected_emb.sin()
print("test_case_1 before assert - cos:")
print(cos)
print("test_case_1 expected cos:")
print(expected_cos)
print("test_case_1 before assert - sin:")
print(sin)
print("test_case_1 expected sin:")
print(expected_sin)
torch.testing.assert_close(cos, expected_cos)
torch.testing.assert_close(sin, expected_sin)
print("test_case_1 passed")


test_case_1 before assert - cos:
tensor([[[ 1.0000,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.9999,  0.5403,  0.9999],
         [-0.4161,  0.9998, -0.4161,  0.9998]]])
test_case_1 expected cos:
tensor([[[ 1.0000,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.9999,  0.5403,  0.9999],
         [-0.4161,  0.9998, -0.4161,  0.9998]]])
test_case_1 before assert - sin:
tensor([[[0.0000, 0.0000, 0.0000, 0.0000],
         [0.8415, 0.0100, 0.8415, 0.0100],
         [0.9093, 0.0200, 0.9093, 0.0200]]])
test_case_1 expected sin:
tensor([[[0.0000, 0.0000, 0.0000, 0.0000],
         [0.8415, 0.0100, 0.8415, 0.0100],
         [0.9093, 0.0200, 0.9093, 0.0200]]])
test_case_1 passed


In [14]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., :x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2,x1),dim = -1)
def apply_rotary_pos_emb(
        q: torch.Tensor,
        k: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        unsqueeze_dim: int = 1,
    ) -> tuple[torch.Tensor, torch.Tensor]:
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim) #[B,1,L,D]
    q_embed = q * cos + rotate_half(q) * sin
    k_embed = k * cos + rotate_half(k) * sin
    return q_embed, k_embed

def forward(
        self,
        x: torch.Tensor, # [B,H,L,D]
        position_ids: torch.Tensor, # [B, L]
        q: Optional[torch.Tensor] = None,
        k: Optional[torch.Tensor] = None,
        unsqueeze_dim: int = 1,
    ):
    B = position_ids.shape[0]
    head_dim = x.shape[-1]
    rope_theta = 10000.0
    inv_freq = 1.0 / (
        rope_theta ** (torch.arange(0,head_dim,2,dtype=torch.float32)/head_dim)
    )  #[D/2]
    inv_freq_expanded = inv_freq[None,:,None].expand(B,-1,1)  #[B,D/2,1]
    position_ids_expanded = position_ids[:,None,:].float() #[B,1,L]
    freqs = (inv_freq_expanded @ position_ids_expanded) #[B,D/2,L]
    freqs = freqs.transpose(1, 2) #[B,L,D/2]
    emb = torch.cat((freqs, freqs), dim=-1) #[B,L,D]
    cos = emb.cos()
    sin = emb.sin()
    q_embed, k_embed = apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=unsqueeze_dim)
    return q_embed, k_embed, cos, sin

    
    
#Rotary Positional Embeddings


In [15]:
# Test case 2: compare your handwritten implementation with the standard implementation above
config = make_config()
rope = LocalLlamaRoPE(config)
B = 1
H = config.num_attention_heads
L = 3
D = config.hidden_size // config.num_attention_heads

x = torch.zeros(B, H, L, D, dtype=torch.float32)
position_ids = torch.tensor([[0, 1, 2]])
q = torch.tensor([[[[1.0, 2.0, 3.0, 4.0], [0.5, -1.0, 1.5, 2.0], [1.0, 0.0, -1.0, 2.0]], [[2.0, -1.0, 0.0, 1.0], [1.0, 1.0, -1.0, -1.0], [0.0, 0.5, 1.0, 1.5]]]])
k = torch.tensor([[[[2.0, 1.0, 0.0, -1.0], [1.0, 1.0, 1.0, 1.0], [-2.0, 0.5, 3.0, -0.5]], [[1.0, 0.0, -2.0, 2.0], [0.0, -1.0, 0.5, 1.5], [2.0, 2.0, -1.0, 0.0]]]])

std_q, std_k, std_cos, std_sin = rope(x, position_ids, q=q, k=k)
my_q, my_k, my_cos, my_sin = forward(None, x, position_ids, q=q, k=k)

print("standard q_embed:")
print(std_q)
print("my q_embed:")
print(my_q)
print("q diff:")
print(my_q - std_q)

print("standard k_embed:")
print(std_k)
print("my k_embed:")
print(my_k)
print("k diff:")
print(my_k - std_k)

print("standard cos:")
print(std_cos)
print("my cos:")
print(my_cos)
print("cos diff:")
print(my_cos - std_cos)

print("standard sin:")
print(std_sin)
print("my sin:")
print(my_sin)
print("sin diff:")
print(my_sin - std_sin)

torch.testing.assert_close(my_q, std_q)
torch.testing.assert_close(my_k, std_k)
torch.testing.assert_close(my_cos, std_cos)
torch.testing.assert_close(my_sin, std_sin)
print("test_case_2 passed")


standard q_embed:
tensor([[[[ 1.0000,  2.0000,  3.0000,  4.0000],
          [-0.9921, -1.0199,  1.2312,  1.9899],
          [ 0.4932, -0.0400,  1.3254,  1.9996]],

         [[ 2.0000, -1.0000,  0.0000,  1.0000],
          [ 1.3818,  1.0099,  0.3012, -0.9900],
          [-0.9093,  0.4699, -0.4161,  1.5097]]]])
my q_embed:
tensor([[[[ 1.0000,  2.0000,  3.0000,  4.0000],
          [-0.9921, -1.0199,  1.2312,  1.9899],
          [ 0.4932, -0.0400,  1.3254,  1.9996]],

         [[ 2.0000, -1.0000,  0.0000,  1.0000],
          [ 1.3818,  1.0099,  0.3012, -0.9900],
          [-0.9093,  0.4699, -0.4161,  1.5097]]]])
q diff:
tensor([[[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],

         [[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]])
standard k_embed:
tensor([[[[ 2.0000,  1.0000,  0.0000, -1.0000],
          [-0.3012,  0.9900,  1.3818,  1.0099],
          [-1.8956,  0.5099, -3.0670, -0.4899]],

         [[ 1.0000,  0.0000, -2.0000,  